In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.getLogger("torch_numopt").setLevel(logging.DEBUG)
# n_epoch = 1000
n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.1 which yielded a loss of 256.797.
DEBUG:torch_numopt.line_search:Iteration 1, new guess is 0.01 which yielded a loss of 0.342027.
INFO:torch_numopt.line_search:Settled into lr = 0.01.
INFO:torch_numopt.scaling_matrix_calculator:Computing the exact hessian matrix.
INFO:torch_numopt.numerical_optimizer:Using matrix form of curvature, solving linear system...
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 4532.359863
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 1719.010742
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 131397.062500
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 47894.718750
DEBUG:torch_numopt.numerical_optimizer:Numerical instabili

epoch:  0, loss: 0.36966490745544434
epoch:  1, loss: 0.34202736616134644


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 5998327.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 1875034.250000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 5344483.500000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1.000000 with method None and initial guess 1.000000
INFO:torch_numopt.line_search:Starting backtracking line search with initial guess of 1 with loss of 7.93321e+09.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.1 which yielded a loss of 10.7837.
DEBUG:torch_numopt.line_search:Iteration 1, new guess is 0.01 which yielded a loss of 0.203834.
INFO:torch_numopt.line_search:Settled into lr = 0.01.
INFO:torch_numopt.scaling_matrix_calculator:Computing the exact hessian matrix.
INFO:torch_numopt.numerical_optimizer:Using matri

epoch:  2, loss: 0.2559413015842438
epoch:  3, loss: 0.20383401215076447
epoch:  4, loss: 0.16865791380405426


INFO:torch_numopt.numerical_optimizer:Using matrix form of curvature, solving linear system...
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 32279448.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 13084005.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 604518848.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 28190864.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 69667758080.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 53040812.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new

epoch:  5, loss: 0.09628494828939438
epoch:  6, loss: 0.09580179303884506


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 4680809472.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 8095870.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 98321280.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1.000000 with method None and initial guess 1.000000
INFO:torch_numopt.line_search:Starting backtracking line search with initial guess of 1 with loss of 3.87386e+06.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.1 which yielded a loss of 0.999518.
DEBUG:torch_numopt.line_search:Iteration 1, new guess is 0.01 which yielded a loss of 0.0764425.
INFO:torch_numopt.line_search:Settled into lr = 0.01.
INFO:torch_numopt.scaling_matrix_calculator:Computing the exact hessian matrix.
INFO:torch_numopt.numerical_optimizer:Using

epoch:  7, loss: 0.08620535582304001
epoch:  8, loss: 0.07644248753786087


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2224065792.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 8180509.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 93158048.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1.000000 with method None and initial guess 1.000000
INFO:torch_numopt.line_search:Starting backtracking line search with initial guess of 1 with loss of 479724.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.1 which yielded a loss of 0.157042.
DEBUG:torch_numopt.line_search:Iteration 1, new guess is 0.01 which yielded a loss of 0.0658191.
INFO:torch_numopt.line_search:Settled into lr = 0.01.


epoch:  9, loss: 0.07632434368133545


In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.2310858964920044
Test metrics:  R2 = 0.10437542200088501


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=0.1,
    line_search_method="interpolate",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

INFO:torch_numopt.scaling_matrix_calculator:Computing the exact hessian matrix.
INFO:torch_numopt.numerical_optimizer:Using matrix form of curvature, solving linear system...
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 23634.859375
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 9316.977539
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 39584.671875
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 45812.433594
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 322633.625000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 555078.000000
DEBUG:torch_numopt.numerical_optimizer

epoch:  0, loss: 0.7349583506584167


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 100772.171875
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 334349.437500
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 786727.250000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 6825149.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2793912.250000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.100000 with method None and initial guess 0.100000
INFO:torch_numopt.line_search:Starting interpolation line search with initial guess of 0.000126814 with loss of 0.679824.
INFO:torch_numopt.line_search:Settled into lr = 0.000126814.
INFO:torch_numopt.scaling_matrix_calculat

epoch:  1, loss: 0.6814703345298767


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 101113.375000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 342873.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 756184.375000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 7541148.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2765334.250000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.100000 with method None and initial guess 0.100000
INFO:torch_numopt.line_search:Starting interpolation line search with initial guess of 3.76834e-05 with loss of 0.679337.
INFO:torch_numopt.line_search:Settled into lr = 3.76834e-05.
INFO:torch_numopt.scaling_matrix_calculat

...

epoch:  7, loss: 0.6766195297241211


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 100893.812500
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 361920.562500
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 730117.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 4725866.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2700358.500000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.100000 with method None and initial guess 0.100000
INFO:torch_numopt.line_search:Starting interpolation line search with initial guess of 5.01822e-05 with loss of 0.675511.
INFO:torch_numopt.line_search:Settled into lr = 5.01822e-05.
INFO:torch_numopt.scaling_matrix_calculat

epoch:  8, loss: 0.6761553287506104


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 100908.656250
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 368961.593750
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 729154.187500
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 5776994.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2689057.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 0.100000 with method None and initial guess 0.100000
INFO:torch_numopt.line_search:Starting interpolation line search with initial guess of 4.58215e-05 with loss of 0.674925.
INFO:torch_numopt.line_search:Settled into lr = 4.58215e-05.


epoch:  9, loss: 0.6755114793777466


In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -29739.408203125
Test metrics:  R2 = -27803.271484375


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="bisect",
    line_search_cond="armijo",
    max_iter=100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

INFO:torch_numopt.scaling_matrix_calculator:Computing the exact hessian matrix.
INFO:torch_numopt.numerical_optimizer:Using matrix form of curvature, solving linear system...
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 7872.800293
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2770.821533
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 7595.790527
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 21965.105469
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 49544.460938
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 130930.921875
DEBUG:torch_numopt.numerical_optimizer:Nu

epoch:  0, loss: 0.0883801132440567


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 2199447.750000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 441431.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 1010089280.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 3882356.250000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 1352460800.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 12919961.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 14985989.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1

epoch:  1, loss: 0.03884536027908325


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 1159203072.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 3112627.750000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 3954419968.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 11329893.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 35138704.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1.000000 with method None and initial guess 1.000000
INFO:torch_numopt.line_search:Starting bisection line search with initial guess of 1 with loss of 1.40894e+08.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.5 which yielded a loss of 291630.
DEBUG:tor

...

epoch:  7, loss: 1166.924560546875


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 350342656.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 187104144.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 151416078336.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 4596654.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 753352835072.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1.000000 with method None and initial guess 1.000000
INFO:torch_numopt.line_search:Starting bisection line search with initial guess of 1 with loss of 14296.4.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.5 which yielded a loss of 102910.
DEBUG:t

epoch:  8, loss: 735.6254272460938


DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 265700640.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 165391024.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 387228467200.000000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 4321394.500000
DEBUG:torch_numopt.numerical_optimizer:Numerical instability found, condition number was inf, new condition number is 704832864256.000000
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1.000000 with method None and initial guess 1.000000
INFO:torch_numopt.line_search:Starting bisection line search with initial guess of 1 with loss of 2383.22.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.5 which yielded a loss of 5901.77.
DEBUG:

epoch:  9, loss: 507.00592041015625


In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -0.0676112174987793
Test metrics:  R2 = -0.013921141624450684
